In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from pathlib import Path


import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import copy

from catboost import CatBoostRegressor

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

import import_ipynb
import os
curdir = os.getcwd()
src_path = Path('../src/')
os.chdir(src_path)
from ML_Base import BaseModelWrapper,  XANESWeightedLoss
os.chdir(curdir)

In [2]:
class LinRegWrapper(BaseModelWrapper):
    def __init__(self, model_type='classic', params=None):
        '''
        model_type: 'classic', 'ridge', или 'lasso'
        params: словарь параметров (например, {'alpha': 1.0} для Ridge/Lasso)
        '''
        self.params = params if params is not None else {}
        self.model_type = model_type.lower()
        self.model = None
        
        self._models = {'classic': LinearRegression, 
                        'ridge': Ridge, 
                        'lasso': Lasso 
                       }

    def fit(self, X_train, y_train, X_val, y_val):
        model_class = self._models.get(self.model_type, LinearRegression)
        self.model = model_class(**self.params)
        self.model.fit(X_train, y_train)
        return [], []

    def predict(self, X):
        return self.model.predict(X)

    def get_feature_importance(self, X_sample=None):
        if self.model is None:
            return None
            
        coef = self.model.coef_
        if coef.ndim > 1:
            return np.mean(np.abs(coef), axis=0)
        return np.abs(coef)

In [3]:
class TorchWrapper(BaseModelWrapper):
    def __init__(self, model, params=None):
        '''
        model: экземпляр nn.Module (MLP, CNN или любая другая архитектура)
        params: словарь с гиперпараметрами (lr, weight_decay, batch_size, patience и т.д.)
        '''
        super().__init__(params)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.model = model.to(self.device)
        
        # Параметры обучения
        lr = self.params.get('lr', 1e-3)
        wd = self.params.get('weight_decay', 1e-3)
        
        self.criterion = XANESWeightedLoss(**self.params.get('loss_params', {'alpha':1.0, 'beta':0., 'gamma':0.})) #MSE
        self.optimizer = optim.AdamW(self.model.parameters(), lr=lr, weight_decay=wd)

        # Настройка планировщика
        T_0 = self.params.get('T_0', 10) # T_0 — количество итераций до первого рестарта
        T_mult = self.params.get('T_mult',2) # T_mult — множитель, увеличивающий длительность каждого следующего цикла
        eta_min = self.params.get('eta_min', 1e-6) # eta_min — минимальное значение, до которого опустится LR
        self.scheduler = CosineAnnealingWarmRestarts(self.optimizer, T_0=T_0, T_mult=T_mult, eta_min=eta_min)
        
        # Параметры ранней остановки
        self.patience = self.params.get('patience', 10)
        self.min_delta = self.params.get('min_delta', 0.0)
        self.best_model_state = None

    def fit(self, X_train, y_train, X_val, y_val):
        # Подготовка данных
        train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train).view(-1, 1))
        train_loader = DataLoader(train_ds, 
                                  batch_size=self.params.get('batch_size', 64), 
                                  shuffle=True
                                 )
        
        X_v = torch.FloatTensor(X_val).to(self.device)
        y_v = torch.FloatTensor(y_val).view(-1, 1).to(self.device)
        
        t_hist, v_hist = [], []
        best_v_loss = float('inf')
        counter = 0
        
        for epoch in range(self.params.get('epochs', 100)):
            self.model.train()
            e_loss = 0
            for bx, by in train_loader:
                bx, by = bx.to(self.device), by.to(self.device)
                self.optimizer.zero_grad()
                
                outputs = self.model(bx)
                loss = self.criterion(outputs, by)
                
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
                e_loss += loss.item()
            
            epoch_train_loss = e_loss / len(train_loader)
            
            # Валидация
            self.model.eval()
            with torch.no_grad():
                v_loss = self.criterion(self.model(X_v), y_v).item()
            
            t_hist.append(epoch_train_loss)
            v_hist.append(v_loss)

            # Логика ранней остановки
            if v_loss < (best_v_loss - self.min_delta):
                best_v_loss = v_loss
                self.best_model_state = copy.deepcopy(self.model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= self.patience:
                    break
        
        if self.best_model_state:
            self.model.load_state_dict(self.best_model_state)
            
        return t_hist, v_hist

    def predict(self, X):
        self.model.eval()
        with torch.no_grad():
            x_tensor = torch.FloatTensor(X).to(self.device)
            if x_tensor.ndim == 1:
                x_tensor = x_tensor.unsqueeze(0)
            
            preds = self.model(x_tensor).cpu().numpy()
            
            # Если на выходе только один таргет (например, CN1), 
            # схлопываем (N, 1) -> (N,)
            if preds.shape[1] == 1:
                return preds.ravel() 
            return preds
            
    def get_feature_importance(self, X_sample):
            '''
            Saliency Map: Градиент выхода по входному спектру.
            Показывает, на какие участки спектра (энергии) модель обращает больше внимания.
            '''
            self.model.eval()
            
            x = torch.FloatTensor(X_sample).to(self.device)
            
            # Если пришел один вектор, делаем из него батч (1, input_dim)
            if x.ndim == 1:
                x = x.unsqueeze(0)
                   
            x.requires_grad_(True)

            output = self.model(x)
            
            # Если выход многомерный (например, предсказание 4-х CN сразу), 
            # берем сумму или градиент по конкретному выходу. Здесь берем сумму.
            output.backward(torch.ones_like(output))
            
            # Извлекаем градиент и берем абсолютное значение
            importance = x.grad.abs().squeeze().cpu().numpy()
            
            return importance

In [4]:
class CatBoostWrapper(BaseModelWrapper):
    def __init__(self, params=None):
        # Параметры по умолчанию
        default_params = {'loss_function': 'RMSE', 
                          'verbose': False, 
                          'allow_writing_files': False, 
                          'early_stopping_rounds': 50, 
                          'iterations': 1000 
                         }

        if params:
            default_params.update(params)
        self.params = default_params
        self.model = None

    def fit(self, X_train, y_train, X_val, y_val):
        self.model = CatBoostRegressor(**self.params)
        
        # Передаем eval_set для контроля переобучения
        self.model.fit(X_train, y_train, 
                       eval_set=(X_val, y_val),
                       use_best_model=True
                      )
        
        # Извлекаем историю метрик
        eval_metrics = self.model.get_evals_result()
        # CatBoost возвращает списки метрик для 'learn' и 'validation'
        train_hist = eval_metrics['learn']['RMSE']
        val_hist = eval_metrics['validation']['RMSE']
        
        return train_hist, val_hist

    def predict(self, X):
        return self.model.predict(X)

    def get_feature_importance(self, X_sample=None):
        if self.model:
            return self.model.get_feature_importance()
        return None

In [5]:
class TreeBaggingWrapper(BaseModelWrapper):
    def __init__(self, model_type='rf', params=None):
        '''
        model_type: 'rf' (RandomForest) или 'et' (ExtraTrees)
        params: словарь с пользовательскими параметрами
        '''
        default_params = {'n_estimators': 100, 
                          'random_state': 42, 
                          'n_jobs': -1 
                         }
        if params:
            default_params.update(params)
            
        self.params = default_params
        self.model_type = model_type.lower()
        self.model = None
        
        self._models = {'rf': RandomForestRegressor, 
                        'et': ExtraTreesRegressor
                       }

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        model_class = self._models.get(self.model_type, RandomForestRegressor)
        self.model = model_class(**self.params)
        self.model.fit(X_train, y_train)
        return [], []


    def predict(self, X):
        return self.model.predict(X)

    def get_feature_importance(self, X_sample=None):
        if self.model:
            return self.model.feature_importances_
        return None